# Tests

## 0 Preliminaries <a id='preliminaries'></a>
### 0.1 config <a id='packages'></a>
[Back to content](#content)

In [1]:
sigma_L = 0.1
Nlens = 1e4

In [2]:
import numpy as np
import sys, platform, os

#the range over which to compute things (used in part one to create params.txt)
sigL_lower = -3
sigL_upper = 1
sigL_n = 90

Nlens_lower = 1
Nlens_upper = 8
Nlens_n = 90

compute_correlations = True     #computes the correlation functions necessary for calculating the covariance matrices,
                                 #and saves them in a pickle file. Set to False if this has already been done, which
                                 #saves computation time
                                 #Note, this is automatically set to True if the correlations were calculated for different
                                 #redshift binning, but NB if the number of bins stays the same but the binscheme changes,
                                 #it should be manually set to True to ensure the old correlations are recalculated

##############################################################################################################
############################################# Survey stats ###################################################
##############################################################################################################

sky_coverage = 15e3     #area of sky covered by the survey (deg^2)

NGal = 2e9              #total number of galaxies (expect 2e9)

######################################## Redshift distribution ##########################################

zmax_dist = 3           #the default maximum redshift being considered
Nbin_z = 1              #the default number of redshift bins

#automatically calculate the redshift bin limits
Nbinz_E = Nbin_z        #the number of redshift bins for galaxy shapes
Nbinz_P = Nbin_z      #the number of redshift bins for galaxy positions

binscheme_E = Nbinz_E
binscheme_P = Nbinz_P

zmax_E = zmax_dist
zmax_P = zmax_dist

#################################### Angles for correlation functions ##################################

Thetamin_arcmin = 0.1                            #minimum theta from which we calculate correlation functions (in arcmin)

Thetamin = Thetamin_arcmin / (60 * 180 / np.pi)  #minimum theta from which we calculate correlation functions (in radians)
Thetamax = np.pi                                 #maximum theta to which we calculate correlation functions (in radians)
nTheta = 10000                                   #number of points used to compute the correlation function

lmax = 1e8
nl = 1000

######################################## Angles for interpolations ##########################################

theta_max_interpolation_arcmin = 300 #the maximum theta that we interpolate to (arcminutes)

theta_min_interpolation = Thetamin #we're working in log space, so we don't start from 0
theta_max_interpolation = theta_max_interpolation_arcmin / (60 * 180 / np.pi)
theta_res_interpolation = 80 #the resolution of thetas in the interpolation (ideally divisible by 16)

Thetamax_LL = theta_max_interpolation          
Thetamax_LE = theta_max_interpolation            
Thetamax_LP = theta_max_interpolation

Thetamax_LL_plus = Thetamax_LL
Thetamax_LL_minus = Thetamax_LL

Thetamax_LE_plus = Thetamax_LE
Thetamax_LE_minus = Thetamax_LE
    
Omegatot = sky_coverage * (np.pi / 180)**2                                #sky coverage (in rad^2)
n_b = ( NGal / sky_coverage * (np.pi / 180)**2 ) / Nbin_z                 #number density of galaxies per redshift bin (in rad^-2)
r2_max = np.sqrt(Omegatot/np.pi)                                          #the maximum theta used in integrals which would normally run from 0 to infty 

########################################## cosmology #########################################################

H0=67.37       #Hubble constant
h=H0/100
ombh2=0.0223   #baryon density parameter
omch2=0.1198   #dark matter density parameter
ns = 0.965     #primordial power spectrum
c = 3e8        #speed of light

Omega_M = (ombh2 + omch2)/(h**2)
Omega_L = 1 - Omega_M

zmax = 7     #the maximum redshift to which we compute the Weyl power spectrum
kmax = 5e2 #(inverse Mpc) the maximum wavenumber, ie. the smallest spatial scales to which we determine the power spectrum
extrap_kmax = 1e10 #the maximum k for extrapolation beyond kmax
chimin = 1e-5

############################################ noise ###########################################################

sigma_E = np.sqrt(2) * 0.3                                            #the noise from the galaxy shapes on cosmic shear

##################################### numerical stuff ########################################################

max_cpus = 512
nsamp_string = '1e3'
nsamp = int(float(nsamp_string))
Csamp = nsamp*10        #default number of samples in the Monte Carlo integrator for triple cosmic integrals
Nsamp = nsamp           #default number of samples in the Monte Carlo integrator for double noise/sparsity integrals
num_batches = 1000     #should be > maxsamp * 373 / (ram per node)
desired_error = 1       #percentage desired fractional error in integrals
confidence = 0.95       #confidence level for the error estimate (not currently used)
warning_level = 500     #level above which we print an integration error
total_error_threshold = 0.2  #the threshold for the total error on a term to be too high and to print a warning

################################################## Imports #####################################################

import matplotlib
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
from tqdm import tqdm
import time

from scipy import constants, special, integrate, stats
from scipy.interpolate import CubicSpline, RegularGridInterpolator
from hankel import HankelTransform

camb_path = os.path.realpath(os.path.join(os.getcwd(),'..'))
sys.path.insert(0,camb_path)
import camb
from camb import model, initialpower

import pickle
from multiprocessing import Pool, cpu_count
from multiprocessing import Process, Manager
from itertools import product
import inspect
from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import root_scalar, minimize_scalar

############################################### Cosmology #####################################################

# CAMB parameters
pars = camb.CAMBparams()                              #initialise the CAMBparams object, which contains all cosmological parameters and settings
pars.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2)   #define the cosmological model
pars.InitPower.set_params(ns=ns)                      #set the primordial power spectrum parameters
background = camb.get_background(pars)                #compute the background cosmological evolution

#this gives us an interpolator which can be used to generate the Weyl power spectrum for any range of z and k
Weyl_power_spectra = camb.get_matter_power_interpolator(pars, zmax=zmax, kmax=kmax, zs=None,
hubble_units=False, k_hunit=False, var1=model.Transfer_Weyl, var2=model.Transfer_Weyl, extrap_kmax=extrap_kmax)

############################################## shared variables ##############################################

# Shared variables
global_dict = {}  

####################### the suffix defining the folder names ###############################################

notes = '' #anything particularly unique about a particular run (eg different redshift binning)
correlation_notes = ''    #needed only to specify that a particular binscheme has been used

def format_sci(n):
    return f'{n:.0e}'.replace('+00', '').replace('+0', '').replace('+', '').replace('-0', '-') 

if not os.path.exists(f'correlations_NE={Nbinz_E}_NP={Nbinz_P}{correlation_notes}'):
    compute_correlations = True      

### 0.2 useful_functions

In [3]:
def radtoarcmin(angle_rad):
    """
    This function converts an an angle expressed in radians
    into arcmins.
    """
    
    angle_arcmin = angle_rad * 60 * 180 / np.pi
    
    return angle_arcmin


def arcmintorad(angle_arcmin):
    """
    This function converts an an angle expressed in arcmins
    into radians.
    """
    
    angle_rad = angle_arcmin / (60 * 180 / np.pi)
    
    return angle_rad

def delta_func(a,b):
    if a == b:
        x = 1
    else:
        x = 0

    return x

def sin2(x):
    return np.sin(2*x)

def cos2(x):
    return np.cos(2*x)


def annuli_intersection_area(i1, o1, i2, o2):
    """
    Computes the intersection area between two concentric annuli.
    
    Parameters:
        i1, o1 : float - Inner and outer radii of first annulus
        i2, o2 : float - Inner and outer radii of second annulus

    Returns:
        float - Area of intersection (0 if no intersection)
    """
    # Compute the overlapping radial range
    r_inner = max(i1, i2)
    r_outer = min(o1, o2)

    if r_outer <= r_inner:
        return 0.0  # No overlap

    # Area of the overlapping annulus
    area = np.pi * (r_outer**2 - r_inner**2)
    
    return area

####################################### printing ############################################

def roundsf(x, sig_figs=1):
    if x == 0:
        return 0  # Avoid log10 issues
    power = math.floor(math.log10(abs(x)))  # Find order of magnitude
    factor = 10 ** power  # Get the scaling factor
    return round(x / factor) * factor  # Round and scale back

############################# file saving and dictionary reading #############################

def save_pickle(data, filename, descriptor):
    """Helper function to save data to a pickle file, creating folders if needed."""
    try:
        dirpath = os.path.dirname(filename)
        if dirpath:
            os.makedirs(dirpath, exist_ok=True)

        with open(filename, "wb") as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        # print(f"Successfully saved {descriptor} at {filename}")
    except Exception as ex:
        print(f"Error during pickling {descriptor}: {ex}")

def add_dict(*objects):
    """Automatically adds multiple objects to global_dict with their variable names as keys."""
    frame = inspect.currentframe().f_back
    for name, value in frame.f_locals.items():
        if any(value is obj for obj in objects):  # Use identity check
            global_dict[name] = value  # Add it to global_dict

def get_item(*names):
    """Retrieves items from global_dict and defines them as global variables in the caller's module."""
    frame = inspect.currentframe().f_back  # Get caller's frame
    caller_globals = frame.f_globals  # Access the caller's global namespace
    
    for name in names:
        if name in global_dict:
            caller_globals[name] = global_dict[name]  # Define the variable globally
        else:
            raise KeyError(f"'{name}' not found in global_dict")

def load_correlations(filename="correlations"):
    """Loads a pickled dictionary from 'filename' and adds its contents to global_dict."""
    try:
        with open(filename, "rb") as f:
            data = pickle.load(f)  # Load the dictionary from the pickle file
    except FileNotFoundError:
        raise FileNotFoundError(f"File '{filename}' not found.")
    except pickle.UnpicklingError:
        raise ValueError(f"File '{filename}' is not a valid pickle file.")

    if isinstance(data, dict):
        global_dict.update(data)  # Merge the loaded dictionary into global_dict
    else:
        raise ValueError("The pickled file does not contain a dictionary.")

def load_file(filename):
    """Loads a pickled dictionary"""
    with open(filename, "rb") as f:
        data = pickle.load(f)  # Load the dictionary from the pickle file

    return(data)

############################### integrals, antiderivatives and solvers ######################################

def radial_integration(correlation_function, Theta_start, Theta_end):

    integrand = lambda x: x*correlation_function(x)
    
    integral, err = quad(integrand, Theta_start, Theta_end)

    return integral
    
def compute_antiderivative(function, theta_max = theta_max_interpolation):

    Thetas = np.logspace(np.log10(theta_min_interpolation), np.log10(theta_max_interpolation), theta_res_interpolation)
    Thetas = np.insert(Thetas, 0, 0.0)  # Prepend 0 to the array
    
    antiderivative_list = [0]
    
    for i in range(theta_res_interpolation):
        
        antiderivative = radial_integration(function, 0, Thetas[i + 1])
        antiderivative_list.append(antiderivative)

    return CubicSpline(Thetas, antiderivative_list)

def find_maximum(f, a, b):
    result = minimize_scalar(lambda x: -f(x), bounds=(a, b), method='bounded')
    if result.success:
        x_max = result.x
        f_max = f(x_max)
        return x_max, f_max
    else:
        raise RuntimeError("Failed to find maximum.")

###################################### Monte Carlo Integrator #######################################

def monte_carlo_integrate(funcs, bounds, num_samples=nsamp, num_batches = num_batches):
    """
    Monte Carlo integration over a given domain with error estimation.
    
    Parameters:
    - funcs (callable or list of callables): The function to integrate. NB it needs to accept an array as input
                       for integration over multiple dimensions.
    - bounds (list of tuples): Integration bounds [(a1, b1), (a2, b2), ...].
                                For 1D, use [(a, b)].
    - num_samples (int): Total number of random samples to be used in the integration
    - num_batches: The number of batches into which our samples are split, to reduce the memory burden
    
    Returns:
    - tuple: (float, float) Estimated value of the integral and its error (or a list of tuples if multiple functions inputted)
    """

    #our function is defined to handle multiple integrands evaluated with the same sample of points. 
    #if a single function is provided, we redefine it as a single-item list
    if not isinstance(funcs, (list, tuple)):
        funcs = [funcs]
        single_function = True
    else:
        single_function = False
    
    nsamp_use = int(num_samples) #the total number of samples in the integration
    batch_size = nsamp_use // num_batches #the number of samples per batch (to reduce the memory burden)
    dim = len(bounds) #the dimensions of the integral
    volumes = [b - a for a, b in bounds] #the volume over which each variable in the integral is evaluated
    total_volume = np.prod(volumes) #the total volume spanned by the integration bounds
    rng = np.random.default_rng() #defining a random number generator

    #define lists of lists, with as many lists as we have functions
    batch_sums = [ [] for _ in funcs ]     # will store the sums of f from each batch
    batch_sumsq = [ [] for _ in funcs ]    # will store the sums of f**2 from each batch

    batch_ns = []                          # will store the number of samples per batch
    #as currently written, every element in batch_ns will simply be equal to batch_size

    #compute rescale values once at the start for each function (for numerical stability)
    n_subsample = 100
    subsamples = np.array([rng.uniform(low=a, high=b, size=n_subsample) for a, b in bounds])
    rescale_vals = []
    for func in funcs:
        f_subsample = func(subsamples)
        typical_scale = np.median(np.abs(f_subsample))
        if typical_scale == 0:
            rescale_vals.append(1.0)  # function is zero, no rescaling needed
        else:
            rescale_vals.append(1.0 / typical_scale)

    #proceed batch by batch
    for _ in range(num_batches):

        #draw a random sample of N points on the integration domain (n=batch_size)
        samples = np.array([rng.uniform(low=a, high=b, size=batch_size) for a, b in bounds])

        #proceed one function at a time
        for i, func in enumerate(funcs):

            values = func(samples) * rescale_vals[i]  #evaluate the function at each of the N sampled points

            batch_sums[i].append(np.sum(values)) #store the sum of each of the n function outputs
            batch_sumsq[i].append(np.sum(values**2)) #store the sum of the square of these outputs
        batch_ns.append(values.size)   #store n (should always be batch_size with the current structure)

    final_integrals = [] #will store the final estimated integrals for each function
    errors = [] #will store the errors in those estimates

    for i in range(len(funcs)):
        N = int(np.sum(batch_ns))                        # total number of samples (might be less than nsamp, depending on the batching)
        total_sum = float(np.sum(batch_sums[i]))         # sum of f over all samples
        total_sumsq = float(np.sum(batch_sumsq[i]))      # sum of f^2 over all samples

        mean_f = total_sum / N #the mean of the function across all points

        if N > 1:
            # sample variance
            var_f = (total_sumsq - N * mean_f**2) / (N - 1) #the sample variance
            var_f = max(var_f, 0.0) #in case of numerical errors leading to negative values
        else:
            var_f = 0.0 #no variance if we've just evaluated at a single point

        mean_integral = total_volume * mean_f #the monte carlo integral estimate for this function
        std_error = total_volume * np.sqrt(var_f / N) #the error in the monte carlo integral for this function

        final_integrals.append(mean_integral/rescale_vals[i])
        errors.append(std_error/rescale_vals[i])

    if single_function:
        return final_integrals[0], errors[0]
        
    else:
        return final_integrals, errors


def test_err(error, signal, name):

    if signal != 0:
        if np.abs(error/signal) > total_error_threshold:
            print(f"Warning! Total error for {name} is {round(np.abs(error/signal),1)}") 

### 0.4 loading the correlations

In [5]:
load_correlations(filename=f"correlations_NE={Nbin_z}_NP={Nbin_z}")

### 0.5 LELE

In [6]:
get_item('LLp','LLx', 'LEp', 'LEx', 'EEp', 'EEx', 'redshift_distributions', 'L0', 'E0')

################################################## LELE cosmic covariance ##############################################################

def generate_ccov_LELE(angular_distribution):
    """
    Computes the contribution of cosmic variance in the covariance matrix
    of the LOS shear - galaxy shape correlation functions.
    """
    
    B = 0
    D = B #only one redshift bin
    
    Nbin       = angular_distribution.Nbina      #the number of angular bins for LE 
    Omegas     = angular_distribution.Omegas     #\Omega_a in the math - the solid angle of bin a (in rad^2)
    rs         = angular_distribution.limits     #the angular bin limits for LE (in rad)

    # Define the integrands
    
    def integrand_pp(params):
        
        psi_b, psi_kd, r_b, r_kd, r_k = params
    
        x_bd = r_kd * np.cos(psi_kd) - r_b * np.cos(psi_b) + r_k
        y_bd = r_kd * np.sin(psi_kd) - r_b * np.sin(psi_b)
        
        r_bd = np.sqrt( x_bd**2 + y_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)

        f = ( ( LLp(r_k) * cos2(psi_b) * cos2(psi_kd)
              + LLx(r_k) * sin2(psi_b) * sin2(psi_kd) )
        * ( EEp[B][D](r_bd) * cos2(psi_bd - psi_b) * cos2(psi_bd - psi_kd)
           + EEx[B][D](r_bd) * sin2(psi_bd - psi_b) * sin2(psi_bd - psi_kd) )
        + ( LEp[D](r_k) * cos2(psi_b) * cos2(psi_kd)
           + LEx[D](r_k) * sin2(psi_b) * sin2(psi_kd) )
        * ( LEp[B](r_bd) * cos2(psi_bd - psi_b) * cos2(psi_bd - psi_kd)
           + LEx[B](r_bd) * sin2(psi_bd - psi_b) * sin2(psi_bd - psi_kd) )
            )
        
        f *= 2 * np.pi * r_k * r_b * r_kd
        
        return f
    
    def integrand_px(params):
        
        psi_b, psi_kd, r_b, r_kd, r_k = params
    
        x_bd = r_kd * np.cos(psi_kd) - r_b * np.cos(psi_b) + r_k
        y_bd = r_kd * np.sin(psi_kd) - r_b * np.sin(psi_b)
        
        r_bd = np.sqrt( x_bd**2 + y_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)

        f = - ( ( LLp(r_k) * cos2(psi_b) * sin2(psi_kd)
              - LLx(r_k) * sin2(psi_b) * cos2(psi_kd) )
        * ( EEp[B][D](r_bd) * cos2(psi_bd - psi_b) * sin2(psi_bd - psi_kd)
           - EEx[B][D](r_bd) * sin2(psi_bd - psi_b) * cos2(psi_bd - psi_kd) )
        + ( LEp[D](r_k) * cos2(psi_b) * sin2(psi_kd)
           - LEx[D](r_k) * sin2(psi_b) * cos2(psi_kd) )
            * ( LEp[B](r_bd) * cos2(psi_bd - psi_b) * sin2(psi_bd - psi_kd)
              - LEx[B](r_bd) * sin2(psi_bd - psi_b) * cos2(psi_bd - psi_kd) )
            )
        
        f *= 2 * np.pi * r_k * r_b * r_kd
        
        return f
    
    def integrand_xp(params):
        
        psi_b, psi_kd, r_b, r_kd, r_k = params
    
        x_bd = r_kd * np.cos(psi_kd) - r_b * np.cos(psi_b) + r_k
        y_bd = r_kd * np.sin(psi_kd) - r_b * np.sin(psi_b)
        
        r_bd = np.sqrt( x_bd**2 + y_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)

        f = - ( ( LLp(r_k) * sin2(psi_b) * cos2(psi_kd)
              - LLx(r_k) * cos2(psi_b) * sin2(psi_kd) )
        * ( EEp[B][D](r_bd) * sin2(psi_bd - psi_b) * cos2(psi_bd - psi_kd)
           - EEx[B][D](r_bd) * cos2(psi_bd - psi_b) * sin2(psi_bd - psi_kd) )
        + ( LEp[D](r_k) * sin2(psi_b) * cos2(psi_kd)
           - LEx[D](r_k) * cos2(psi_b) * sin2(psi_kd) )
            * ( LEp[B](r_bd) * sin2(psi_bd - psi_b) * cos2(psi_bd - psi_kd)
              - LEx[B](r_bd) * cos2(psi_bd - psi_b) * sin2(psi_bd - psi_kd) )
            )
        
        f *= 2 * np.pi * r_k * r_b * r_kd
        
        return f
    
    def integrand_xx(params):
        
        psi_b, psi_kd, r_b, r_kd, r_k = params
    
        x_bd = r_kd * np.cos(psi_kd) - r_b * np.cos(psi_b) + r_k
        y_bd = r_kd * np.sin(psi_kd) - r_b * np.sin(psi_b)
        
        r_bd = np.sqrt( x_bd**2 + y_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)

        f = ( ( LLp(r_k) * sin2(psi_b) * sin2(psi_kd)
              + LLx(r_k) * cos2(psi_b) * cos2(psi_kd) )
            * ( EEp[B][D](r_bd) * sin2(psi_bd - psi_b) * sin2(psi_bd - psi_kd)
             + EEx[B][D](r_bd) * cos2(psi_bd - psi_b) * cos2(psi_bd - psi_kd) )
            + ( LEp[D](r_k) * sin2(psi_b) * sin2(psi_kd)
              + LEx[D](r_k) * cos2(psi_b) * cos2(psi_kd) )
            * ( LEp[B](r_bd) * sin2(psi_bd - psi_b) * sin2(psi_bd - psi_kd)
              + LEx[B](r_bd) * cos2(psi_bd - psi_b) * cos2(psi_bd - psi_kd) )
            )
        
        f *= 2 * np.pi * r_k * r_b * r_kd

        return f
        
    def integral_bins(integrand, alpha, beta):
        
        ranges = [(0, 2*np.pi), (0, 2*np.pi),
                  (rs[alpha], rs[alpha+1]), (rs[beta], rs[beta+1]), (0, r2_max)]
        
        integral, err = monte_carlo_integrate(integrand, ranges, Csamp)
        
        # normalisation of differential elements
        integral /= (Omegatot * Omegas[alpha] * Omegas[beta]) 
        err /= (Omegatot * Omegas[alpha] * Omegas[beta]) 
        return integral, err
            
    ccov_pp, err_pp = integral_bins(integrand_pp, 0, 0)
    ccov_px, err_px = integral_bins(integrand_px, 0, 0)
    ccov_xp, err_xp = integral_bins(integrand_xp, 0, 0)
    ccov_xx, err_xx = integral_bins(integrand_xx, 0, 0)

    err = np.sqrt(err_pp**2 + err_px**2 + err_xp**2 + err_xx**2)

    ccovpp = ccov_pp + ccov_px + ccov_xp + ccov_xx
    ccovmm = ccov_pp - ccov_px - ccov_xp + ccov_xx
    
    return ccovpp, ccovmm, err

def LELE_ccov_v_theta(theta):
    
    angular_distribution = Angular_Distributions(binscheme=[0,theta], Nbin_a=1, Thetamax=None)
    ccovpp, ccovmm, err = generate_ccov_LELE(angular_distribution)

    return ccovpp, ccovmm

################################################## LELE noise/sparsity covariance #############################################################

def LELE_ncov_v_theta(theta):
    """
    Computes the contribution of noise and sparsity variance in the covariance matrix
    of the LOS shear - galaxy shape correlation functions.
    """
    
    B = 0
    D = B #only one redshift bin

    angular_distribution = Angular_Distributions(binscheme=[0,theta], Nbin_a=1, Thetamax=None)
    D = B #same redshift bin
    
    Nbin       = angular_distribution.Nbina      #the number of angular bins for LE (sign1) 
    Omegas     = angular_distribution.Omegas     #\Omega_a in the math - the solid angle of bin a (in rad^2) (sign1)
    rs         = angular_distribution.limits     #the angular bin limits for LE (in rad) (sign1)
    
    redshift_distribution = redshift_distributions['E']
    
    G_B    = redshift_distribution.get_ngal(B)         #G_B in the math - the number of galaxies in redshift bin B
    
    # Define the integrands
    
    def integrand_pp_L(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = ( cos2(psi_d) 
            * ( EEp[B][D](r_bd) * cos2(psi_bd) * cos2(psi_bd - psi_d) 
              + EEx[B][D](r_bd) * sin2(psi_bd) * sin2(psi_bd - psi_d) 
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    def integrand_pp_E(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = ( cos2(psi_d) 
            * ( LLp(r_bd) * cos2(psi_bd) * cos2(psi_bd - psi_d)  
              + LLx(r_bd) * sin2(psi_bd) * sin2(psi_bd - psi_d)  
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    def integrand_px_L(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = - ( sin2(psi_d) 
            * ( EEp[B][D](r_bd) * cos2(psi_bd) * sin2(psi_bd - psi_d)  
              - EEx[B][D](r_bd) * sin2(psi_bd) * cos2(psi_bd - psi_d)  
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    def integrand_px_E(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = - ( sin2(psi_d) 
            * ( LLp(r_bd) * cos2(psi_bd) * sin2(psi_bd - psi_d) 
              - LLx(r_bd) * sin2(psi_bd) * cos2(psi_bd - psi_d) 
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    def integrand_xp_L(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = ( sin2(psi_d) 
            * ( EEp[B][D](r_bd) * sin2(psi_bd) * cos2(psi_bd - psi_d)  
              - EEx[B][D](r_bd) * cos2(psi_bd) * sin2(psi_bd - psi_d)  
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    def integrand_xp_E(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = ( sin2(psi_d) 
            * ( LLp(r_bd) * sin2(psi_bd) * cos2(psi_bd - psi_d)   
              - LLx(r_bd) * cos2(psi_bd) * sin2(psi_bd - psi_d) 
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    def integrand_xx_L(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = ( cos2(psi_d) 
            * ( EEp[B][D](r_bd) * sin2(psi_bd) * sin2(psi_bd - psi_d)  
              + EEx[B][D](r_bd) * cos2(psi_bd) * cos2(psi_bd - psi_d)  
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    def integrand_xx_E(params):
        
        r_b, r_d, psi_d = params
    
        y_bd = r_d*np.sin(psi_d)
        x_bd = r_d*np.cos(psi_d) - r_b
        
        r_bd = np.sqrt( y_bd**2 + x_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = ( cos2(psi_d) 
            * ( LLp(r_bd) * sin2(psi_bd) * sin2(psi_bd - psi_d) 
              + LLx(r_bd) * cos2(psi_bd) * cos2(psi_bd - psi_d) 
            ) )

        f *= np.pi * r_b * r_d #the usual 2 disappears because of the factor 1/2 out the front
                              
        return f
    
    
    def integral_bins(integrand, alpha, beta):
        
        ranges = [(rs[alpha], rs[alpha+1]), (rs[beta], rs[beta+1]), (0, 2*np.pi)]
        
        integral, err = monte_carlo_integrate(integrand, ranges, Nsamp)
        
        # normalisation of differential elements
        integral /= (Omegas[alpha] * Omegas[beta]) 
        err      /= (Omegas[alpha] * Omegas[beta]) 
        
        return integral, err
    
    integrand_pp = [integrand_pp_L, integrand_pp_E]
    integrand_px = [integrand_px_L, integrand_px_E]
    integrand_xp = [integrand_xp_L, integrand_xp_E]
    integrand_xx = [integrand_xx_L, integrand_xx_E]

    int_pp, err_pp = integral_bins(integrand_pp, 0, 0)
    int_px, err_px = integral_bins(integrand_px, 0, 0)
    int_xp, err_xp = integral_bins(integrand_xp, 0, 0)
    int_xx, err_xx = integral_bins(integrand_xx, 0, 0)

    return int_pp, int_px, int_xp, int_xx

def generate_ncov_LELE(sigma_L, Nlens, angular_distribution):
    """
    Computes the contribution of noise and sparsity variance in the covariance matrix
    of the LOS shear - galaxy shape correlation functions.
    
    B             : the galaxy redshift bin D (0 to Nbinz_E)
    """

    B = 0
    D = B #same redshift bin
    
    Nbin       = angular_distribution.Nbina      #the number of angular bins for LE (sign1) 
    Omegas     = angular_distribution.Omegas     #\Omega_a in the math - the solid angle of bin a (in rad^2) (sign1)
    rs         = angular_distribution.limits     #the angular bin limits for LE (in rad) (sign1)
    
    redshift_distribution = redshift_distributions['E']
    
    G_B    = redshift_distribution.get_ngal(B)         #G_B in the math - the number of galaxies in redshift bin B
    
    # Define the integrands

    ncov_pp = (sigma_L**2/Nlens) * LELE_int_pp[0](rs[1])

    # nerr_pp = (sigma_L**2/Nlens) * err_pp[0]

    scov_pp = (L0/Nlens) * LELE_int_pp[0](rs[1])

    # serr_pp = (L0/Nlens) * err_pp[0]

    ncov_px = (sigma_L**2/Nlens) * LELE_int_px[0](rs[1])

    # nerr_px = (sigma_L**2/Nlens) * err_px[0]

    scov_px = (L0/Nlens) * LELE_int_px[0](rs[1])

    # serr_px = (L0/Nlens) * err_px[0]

    ncov_xp = (sigma_L**2/Nlens) * LELE_int_xp[0](rs[1])

    # nerr_xp = (sigma_L**2/Nlens) * err_xp[0]

    scov_xp = (L0/Nlens) * LELE_int_xp[0](rs[1])

    # serr_xp = (L0/Nlens) * err_xp[0]

    ncov_xx = (sigma_L**2/Nlens) * LELE_int_xx[0](rs[1])

    # nerr_xx = (sigma_L**2/Nlens) * err_xx[0]

    scov_xx = (L0/Nlens) * LELE_int_xx[0](rs[1])

    # serr_xx = (L0/Nlens) * err_xx[0]

    ncov_pp += (sigma_E**2/G_B) * LELE_int_pp[1](rs[1]) 

    # nerr_pp = (
    #          np.sqrt( nerr_pp**2
    #             + ( (sigma_E**2/G_B) * err_pp[1])**2 )
    #           ) 

    scov_pp += (E0[B]/G_B) * LELE_int_pp[1](rs[1]) 

    # serr_pp = (
    #           np.sqrt(  serr_pp**2
    #                   + ( (E0[B]/G_B) * err_pp[1])**2 )
    #           )

    ncov_px += (sigma_E**2/G_B) * LELE_int_px[1](rs[1]) 

    # nerr_px = (
    #           np.sqrt(  nerr_px**2
    #                   + ( (sigma_E**2/G_B) * err_px[1])**2 )
    #           )

    scov_px += (E0[B]/G_B) * LELE_int_px[1](rs[1]) 

    # serr_px = (
    #           np.sqrt(  serr_px**2
    #                   + ( (E0[B]/G_B) * err_px[1])**2 )
    #           )

    ncov_xp += (sigma_E**2/G_B) * LELE_int_xp[1](rs[1]) 

    # nerr_xp = (
    #           np.sqrt(  nerr_xp**2
    #                   + ( (sigma_E**2/G_B) * err_xp[1])**2 )
    #           )

    scov_xp += (E0[B]/G_B) * LELE_int_xp[1](rs[1]) 

    # serr_xp = (
    #           np.sqrt(  serr_xp**2
    #                   + ( (E0[B]/G_B) * err_xp[1])**2 )
    #           )

    ncov_xx += (sigma_E**2/G_B) * LELE_int_xx[1](rs[1]) 

    # nerr_xx = (
    #           np.sqrt(  nerr_xx**2
    #                   + ( (sigma_E**2/G_B) * err_xx[1])**2 )
    #           )

    scov_xx += (E0[B]/G_B) * LELE_int_xx[1](rs[1]) 

    # serr_xx = (
    #           np.sqrt(  serr_xx**2
    #                   + ( (E0[B]/G_B) * err_xx[1])**2 )
    #           )

    cterm_n = ( (1/4) * (1/Nlens) * (1/G_B) 
              * ( sigma_L**2 * sigma_E**2
                + sigma_L**2 * E0[B]
                + L0 * sigma_E**2 ) 
               * Omegatot / Omegas[0] )
    
    cterm_s = ( (1/4) * (1/Nlens) * (1/G_B) 
              * L0 * E0[B] 
               * Omegatot / Omegas[0])

    ncov_pp += cterm_n
    ncov_xx += cterm_n
    
    scov_pp += cterm_s
    scov_xx += cterm_s

    # nerr = np.sqrt(nerr_pp**2 + nerr_px**2 + nerr_xp**2 + nerr_xx**2)
    # serr = np.sqrt(serr_pp**2 + serr_px**2 + serr_xp**2 + serr_xx**2)

    ncovpp = ncov_pp + ncov_px + ncov_xp + ncov_xx    
    scovpp = scov_pp + scov_px + scov_xp + scov_xx
    
    ncovmm = ncov_pp - ncov_px - ncov_xp + ncov_xx
    scovmm = scov_pp - scov_px - scov_xp + scov_xx

    ncov = [ncovpp, ncovmm]
    scov = [scovpp, scovmm]
    
    return [ncov, scov]#, [nerr, serr]


### 0.6 LPLP

In [7]:
get_item('LLp','LLx','LP', 'PP', 'redshift_distributions', 'L0')

################################################## LPLP cosmic covariance ##############################################################
def generate_ccov_LPLP(angular_distribution):
    """
    Computes the contribution of cosmic variance in the covariance matrix
    of the LOS shear - galaxy position correlation functions.
    """
    
    Nbin       = angular_distribution.Nbina      #the number of angular bins for LP (redshift bin B)
    Omegas     = angular_distribution.Omegas     #\Omega_a in the math - the solid angle of bin a (in rad^2) (redshift bin B)
    rs         = angular_distribution.limits     #the angular bin limits for LP (in rad) (redshift bin B)

    B = 0 
    D = B #same redshift bin
    
    # Define the integrands
    
    def integrand(params):
        
        psi_b, psi_kd, r_b, r_kd, r_k = params
        
        x_bd = r_kd * np.cos(psi_kd) - r_b * np.cos(psi_b) + r_k
        y_bd = r_kd * np.sin(psi_kd) - r_b * np.sin(psi_b)
        
        r_bd = np.sqrt( x_bd**2 + y_bd**2 ) 
        psi_bd = np.arctan2(y_bd, x_bd)
        
        f = ( (LLp(r_k) * cos2(psi_b) * cos2(psi_kd)
            + LLx(r_k) * sin2(psi_b) * sin2(psi_kd))
            * PP[B][D](r_bd)
            + LP[D](r_bd) * LP[B](r_k) * cos2(psi_bd-psi_b) * cos2(psi_kd)
            )
        
        f *= 2 * np.pi * r_k * r_b * r_kd
        
        return f
    
    def integral_bins(integrand, alpha, beta):
        
        ranges = [(0, 2*np.pi), (0, 2*np.pi),
                  (rs[alpha], rs[alpha+1]), (rs[beta], rs[beta+1]), (0, r2_max)]
        
        integral, err = monte_carlo_integrate(integrand, ranges, Csamp)
        
        # normalisation of differential elements
        integral /= (Omegatot * Omegas[alpha] * Omegas[beta]) 
        err /= (Omegatot * Omegas[alpha] * Omegas[beta]) 
        return integral, err
    
    ccov, err = integral_bins(integrand, 0, 0)
    
    return ccov, err

def LPLP_ccov_v_theta(theta):
    
    angular_distribution = Angular_Distributions(binscheme=[0,theta], Nbin_a=1, Thetamax=None)
    ccov, err = generate_ccov_LPLP(angular_distribution)

    return ccov

################################################## LPLP noise/sparsity covariance #############################################################

def LPLP_ncov_v_theta(theta):
    """
    Computes the contribution of noise and sparsity variance in the 
    covariance matrix of the LOS shear - galaxy position correlation functions.
    """
    
    angular_distribution = Angular_Distributions(binscheme=[0,theta], Nbin_a=1, Thetamax=None)
    
    B = 0
    D = B #same redshift bin
    
    Nbin       = angular_distribution.Nbina      #the number of angular bins
    Omegas     = angular_distribution.Omegas     #\Omega_a in the math - the solid angle of bin a (in rad^2)
    rs         = angular_distribution.limits     #the angular bin limits (in rad)
    
    redshift_distribution = redshift_distributions['P']

    G_B    = redshift_distribution.get_ngal(B)         #G_B in the math - the number of galaxies in redshift bin B
    
    # Define the integrands
    
    def integrand_P(params):
        
        r_i, r_k, psi_k = params
    
        y_ik = r_k*np.sin(psi_k)
        x_ik = r_k*np.cos(psi_k) - r_i
        
        r_ik = np.sqrt( y_ik**2 + x_ik**2 ) 
        psi_ik = np.arctan2(y_ik, x_ik)
        
        f = ( LLp(r_ik) * cos2(psi_ik) * cos2(psi_ik - psi_k) 
            + LLx(r_ik) * sin2(psi_ik) * sin2(psi_ik - psi_k) )

        f *= 2 * np.pi * r_i * r_k
                              
        return f
    
    def integrand_L(params):
        
        r_i, r_k, psi_k = params
    
        y_ik = r_k*np.sin(psi_k)
        x_ik = r_k*np.cos(psi_k) - r_i
        
        r_ik = np.sqrt( y_ik**2 + x_ik**2 ) 
        psi_ik = np.arctan2(y_ik, x_ik)
        
        f = (1/2) * PP[B][D](r_ik) * cos2(psi_k)

        f *= 2 * np.pi * r_i * r_k
                              
        return f
    
    def integral_bins(integrand, alpha, beta):
        
        ranges = [(rs[alpha], rs[alpha+1]), (rs[beta], rs[beta+1]), (0, 2*np.pi)]
        
        integral, err = monte_carlo_integrate(integrand, ranges, Nsamp)
        
        # normalisation of differential elements
        integral /= (Omegas[alpha] * Omegas[beta]) 
        err     /= (Omegas[alpha] * Omegas[beta]) 
        
        return integral, err

    integrand = [integrand_L, integrand_P]
         
    intt, err = integral_bins(integrand, 0, 0)
             
    return intt

def generate_ncov_LPLP(sigma_L, Nlens, angular_distribution):
    """
    Computes the contribution of noise and sparsity variance in the 
    covariance matrix of the LOS shear - galaxy position correlation functions.
    """

    B = 0
    
    Nbin       = angular_distribution.Nbina      #the number of angular bins
    Omegas     = angular_distribution.Omegas     #\Omega_a in the math - the solid angle of bin a (in rad^2)
    rs         = angular_distribution.limits     #the angular bin limits (in rad)
    
    redshift_distribution = redshift_distributions['P']

    G_B    = redshift_distribution.get_ngal(B)         #G_B in the math - the number of galaxies in redshift bin B
    
    ncov = (sigma_L**2/Nlens) * LPLP_int[0](rs[1])     
    # nerr = (sigma_L**2/Nlens) * err[0]
             
    scov = (L0/Nlens) * LPLP_int[0](rs[1])     
    # serr = (L0/Nlens) * err[0]

    ncov += (1/G_B) * LPLP_int[1](rs[1])  
    scov += (1/G_B) * LPLP_int[1](rs[1])  

    ncov += (1/2) * (sigma_L**2 / (Nlens*G_B) ) * (Omegatot/Omegas[0])
    scov += (1/2) * (L0 / (Nlens*G_B) ) * (Omegatot/Omegas[0])
    
    return [ncov, scov]#, [nerr, serr]

In [8]:
## 0.7 angular_distributions

In [9]:
def optimise_bins(antiderivative, variance):
    
    redshift_distributions = load_file(f"data/redshift_distributions")
    
    def SNR(theta):

        numerator = np.abs(antiderivative(theta) - antiderivative(0))
        
        denom_sq = np.abs(variance(theta)) #this isn't great, need to check why it's like this
        
        if not np.isfinite(denom_sq):
            print(f"[BAD VAR] non-finite at theta={theta}: {denom_sq}", flush=True)
        elif denom_sq <= 0:
            print(f"[BAD VAR] non-positive at theta={theta}: {denom_sq}", flush=True)

        denominator = np.sqrt(denom_sq)

        prefactor = 2/(theta**2)
        
        return prefactor * numerator / denominator

    theta_optimal, SNR_max = find_maximum(SNR, 0, theta_max_interpolation)

    optimised_signal = 2/(theta_optimal**2)*antiderivative(theta_optimal) - antiderivative(0) #need to check
    optimised_noise = np.sqrt(np.abs(variance(theta_optimal))) #this isn't great, need to check why it's like this

    return theta_optimal, optimised_signal, optimised_noise, SNR_max         #return a single bin

class Angular_Distributions:
    """
    This class produces anything useful related to the statistics of lenses and galaxies,
    and their binning in angular separation
    """
    
    def __init__(self, binscheme=None, sky_coverage=sky_coverage, Nbin_a=None, Thetamax=theta_max_interpolation):
        """
        Arguments:
        - Nlens         : number of lenses we can use
        - Ngal          : number of galaxies we can use
        - sky_coverage  : area of the survey footprint, in deg2 
        - Nbin_a         : number of bins of angular separation
        - b1            : the first angular separation bin
        - b2            : the second angular separation bin
        - Thetamax      : maximum angular separation, in rad (only used if no binscheme list supplied)
        - binscheme     : list, min and max angular separations of each angular separation bin
        
        All the angular attributes will be expressed in rad.
        """
        
        # Lens number and density
        self.Omegatot = sky_coverage * (np.pi / 180)**2 # in rad2

        # Binning
        if isinstance(binscheme, int): 
            
            self.Nbina = Nbin_a
            self.Omega = np.pi * Thetamax**2 / Nbin_a
            self.Omegas = self.Omega * np.ones(Nbin_a) # in rad2 
            
            # the list of Omegas is in case we want to have different bin sizes
            self.limits = np.sqrt(self.Omega / np.pi * np.arange(Nbin_a + 1)) # in rad        
            
            # "Centres" of the bins, taken as the mean separation of objects within the angular bin
            self.Thetas = 2/3 * (self.limits[1:]**3  - self.limits[:-1]**3 ) / (self.limits[1:]**2  - self.limits[:-1]**2 ) 
            
        elif isinstance(binscheme, list):
                
            Omegas = []  #the angular size of each bin, in rad^2
            limits = []
            limits.append(binscheme[0])    #the minimum angular separation considered
                
            Nbin_a = len(binscheme) - 1
            self.Nbina = Nbin_a 
                
            for i in range(Nbin_a):
                omega = np.pi * (binscheme[i+1]**2-binscheme[i]**2) #the angular size of that bin, pi x (R_outer^2 - r_inner^2) 
                Omegas.append(omega)
                limits.append(binscheme[i+1])
                
            self.Omegas = np.array(Omegas)
                
            # the list of Omegas is in case we want to have different bin sizes
            self.limits = np.array(limits)        
                
            # "Centres" of the bins, taken as the median separation
            self.Thetas = np.sqrt((self.limits[1:]**2 + self.limits[:-1]**2) / 2) # in rad
        
        else:
            print("Error: no binscheme defined")        

    
    def compute_binned_correlation(self, correlation):
        """
        This function computes the expected signal for a correlation function with this binning
        Note, this is only relevant for autocorrelation functions.
        """
        
        xi_bins  = []
        
        Nbin   = self.Nbina
        rs     = self.limits
        Omegas = self.Omegas

        def integrand(r):
            f = 2 * np.pi * r * correlation(r)
            return f
        
        for a in range(Nbin):
            integral, err = monte_carlo_integrate(integrand, [(rs[a], rs[a+1])])
            integral /= Omegas[a]
            xi_bins.append(integral)

        return xi_bins


def generate_binned_correlation(correlation, b1):
    """Handles generation of binned correlation functions based on cov_matrix type."""
    correlation_data = {}

    get_item('LL_plus','LL_minus')
    get_item('LE_plus','LE_minus')
    get_item('LP')
    get_item('angular_distributions')

    correlation_mapping = {
        "LL": lambda: {
            "plus_correlation": angular_distributions['LL_plus'].compute_binned_correlation(LL_plus),
            "minus_correlation": angular_distributions['LL_minus'].compute_binned_correlation(LL_minus),
        },
        "LE": lambda: {
            "plus_correlation": angular_distributions['LE_plus'][b1].compute_binned_correlation(LE_plus[b1]),
            "minus_correlation": angular_distributions['LE_minus'][b1].compute_binned_correlation(LE_minus[b1]),
        },
        "LP": lambda: {
            "correlation": angular_distributions['LP'][b1].compute_binned_correlation(LP[b1]),
        }
    }

    if correlation in correlation_mapping:
        correlation_data = correlation_mapping[correlation]()
    
    return correlation_data

## 2. Part II

In [12]:
def split_array(array, n_chunks):
        return np.array_split(array, n_chunks)

thetas = np.logspace(np.log10(theta_min_interpolation), np.log10(theta_max_interpolation), theta_res_interpolation)
# thetas = np.insert(thetas, 0, 0.0)  # Prepend 0 to the array

In [13]:
################################################ 2.2 LELE_ccov ###############################################################
##############################################################################################################################

print('LELE_ccov')

def evaluate_chunk(theta_chunk):
    plus = []
    minus = []
    
    for theta in theta_chunk[0]:
        p, m = LELE_ccov_v_theta(theta)
        plus.append(p)
        minus.append(m)
        
    return np.array(plus), np.array(minus)

theta_chunks = split_array(thetas, 1)

results = evaluate_chunk(theta_chunks)

plus_values = results[0]
minus_values = results[1]

LELE_ccov_plus = CubicSpline(thetas, plus_values)
LELE_ccov_minus = CubicSpline(thetas, minus_values)

################################################ 3.2 LELE_ncov ###############################################################
##############################################################################################################################

print('LELE_ncov')

def evaluate_chunk(theta_chunk):
    pp_list = []
    px_list = []
    xp_list = []
    xx_list = []
    
    for theta in theta_chunk[0]:
        pp, px, xp, xx = LELE_ncov_v_theta(theta)
        pp_list.append(pp)
        px_list.append(px)
        xp_list.append(xp)
        xx_list.append(xx)
        
    return np.array(pp_list), np.array(px_list), np.array(xp_list), np.array(xx_list) 

theta_chunks = split_array(thetas, 1)

results = evaluate_chunk(theta_chunks)

intpp_values = results[0]
intpx_values = results[1]
intxp_values = results[2]
intxx_values = results[3]

LELE_int_pp = [CubicSpline(thetas, intpp_values[:, 0]), CubicSpline(thetas, intpp_values[:, 1])]
LELE_int_px = [CubicSpline(thetas, intpx_values[:, 0]), CubicSpline(thetas, intpx_values[:, 1])]
LELE_int_xp = [CubicSpline(thetas, intxp_values[:, 0]), CubicSpline(thetas, intxp_values[:, 1])]
LELE_int_xx = [CubicSpline(thetas, intxx_values[:, 0]), CubicSpline(thetas, intxx_values[:, 1])]

LELE_ccov
LELE_ncov


In [14]:
################################################ 2.3 LPLP_ccov ###############################################################
##############################################################################################################################

print('LPLP_ccov')

def evaluate_chunk(theta_chunk):
    values = []
    
    for theta in theta_chunk[0]:
        v = LPLP_ccov_v_theta(theta)
        values.append(v)
        
    return np.array(values)

theta_chunks = split_array(thetas, 1)

values = evaluate_chunk(theta_chunks)

LPLP_ccov = CubicSpline(thetas, values)

print(LPLP_ccov(0.08))

################################################ 3.3 LPLP_ncov ###############################################################
##############################################################################################################################

print('LPLP_ncov')

def evaluate_chunk(theta_chunk):
    values = []

    for theta in theta_chunk[0]:
        values.append(LPLP_ncov_v_theta(theta))

    return np.array(values)

theta_chunks = split_array(thetas, 1)

int_values = evaluate_chunk(theta_chunks)

LPLP_int = [CubicSpline(thetas, int_values[:, 0]), CubicSpline(thetas, int_values[:, 1])]

LPLP_ccov
1.227619015008789e-12
LPLP_ncov


## 3. Part III

In [15]:
print('LELE')

def LELE_ncov(theta): 
    """ 
    a function which returns the noise and sparsity as a function of theta only (for use in 
    the optimisation). Works for the specific sigma_L and Nlens provided by the system 
    arguments of this run.

    returns [[ncov_plus, ncov_minus], [scov_plus, scov_minus]]
    """
    
    angular_distribution = Angular_Distributions(binscheme=[0,theta], Nbin_a=1, Thetamax=None)

    return generate_ncov_LELE(sigma_L, Nlens, angular_distribution)

######################################################## 3.1 plus ############################################################
##############################################################################################################################

get_item('LE_plus_primitive','LE_minus_primitive','LP_primitive')

print('LELE plus')

def total_variance_LELE_plus(theta):
    """ 
    a function which returns the total LELE plus variance as a function of theta
    (hence for use in the optimisation).
    """

    noise_sparsity = LELE_ncov(theta) #generates the noise and sparsity as a function of theta

    noise = noise_sparsity[0] #extracts [noise_plus, noise_minus]
    sparsity = noise_sparsity[1] #extracts [sparsity_plus, sparsity_minus]

    noise_plus = noise[0] #extracts noise_plus
    sparsity_plus = sparsity[0] #extracts sparsity_plus
    
    variance = noise_plus + sparsity_plus + LELE_ccov_plus(theta).item() #the total variance
    
    return variance

#optimise the binning
LELE_plus_optimised_theta, LELE_plus_optimised_signal, LELE_plus_optimised_std, LELE_plus_optimised_SNR = optimise_bins(LE_plus_primitive[0], total_variance_LELE_plus)

#evaluate the noise and sparsity at the optimised theta
LELE_plus_optimised_noise_sparsity = LELE_ncov(LELE_plus_optimised_theta)

LELE_plus_optimised_noise = LELE_plus_optimised_noise_sparsity[0][0] #extract the optimised noise_plus component
LELE_plus_optimised_sparsity = LELE_plus_optimised_noise_sparsity[1][0] #extract the optimised sparsity_plus component
LELE_plus_optimised_cosmic = LELE_ccov_plus(LELE_plus_optimised_theta).item() #extract the optimised cosmic_plus component

LELE_plus_dict = {'optimised_theta': LELE_plus_optimised_theta,
                  'optimised_signal': LELE_plus_optimised_signal,
                  'optimised_std': LELE_plus_optimised_std,
                  'optimised_SNR': LELE_plus_optimised_SNR,
                  'optimised_noise': LELE_plus_optimised_noise,
                  'optimised_sparsity': LELE_plus_optimised_sparsity,
                  'optimised_cosmic': LELE_plus_optimised_cosmic}

print(LELE_plus_dict)

######################################################## 3.2 minus ############################################################
##############################################################################################################################

print('LELE minus')

def total_variance_LELE_minus(theta):
    """ 
    a function which returns the total LELE minus variance as a function of theta
    (hence for use in the optimisation).
    """

    noise_sparsity = LELE_ncov(theta) #generates the noise and sparsity as a function of theta

    noise = noise_sparsity[0] #extracts [noise_plus, noise_minus]
    sparsity = noise_sparsity[1] #extracts [sparsity_plus, sparsity_minus]

    noise_minus = noise[1] #extracts noise_minus
    sparsity_minus = sparsity[1] #extracts sparsity_minus
    
    variance = noise_minus + sparsity_minus + LELE_ccov_minus(theta).item() #the total variance
    
    return variance

#optimise the binning
LELE_minus_optimised_theta, LELE_minus_optimised_signal, LELE_minus_optimised_std, LELE_minus_optimised_SNR = optimise_bins(LE_minus_primitive[0], total_variance_LELE_minus)

#evaluate the noise and sparsity at the optimised theta
LELE_minus_optimised_noise_sparsity = LELE_ncov(LELE_minus_optimised_theta)

LELE_minus_optimised_noise = LELE_minus_optimised_noise_sparsity[0][1] #extract the optimised noise_minus component
LELE_minus_optimised_sparsity = LELE_minus_optimised_noise_sparsity[1][1] #extract the optimised sparsity_minus component
LELE_minus_optimised_cosmic = LELE_ccov_minus(LELE_minus_optimised_theta).item() #extract the optimised cosmic_minus component

LELE_minus_dict = {'optimised_theta': LELE_minus_optimised_theta,
                  'optimised_signal': LELE_minus_optimised_signal,
                  'optimised_std': LELE_minus_optimised_std,
                  'optimised_SNR': LELE_minus_optimised_SNR,
                  'optimised_noise': LELE_minus_optimised_noise,
                  'optimised_sparsity': LELE_minus_optimised_sparsity,
                  'optimised_cosmic': LELE_minus_optimised_cosmic}

print(LELE_minus_dict)

##############################################################################################################################
######################################################### 4. LPLP ############################################################
##############################################################################################################################

print('LPLP')

def LPLP_ncov(theta): 
    """ 
    a function which returns the noise and sparsity as a function of theta only (for use in 
    the optimisation). Works for the specific sigma_L and Nlens provided by the system 
    arguments of this run.

    returns [ncov, scov]
    """
    
    angular_distribution = Angular_Distributions(binscheme=[0,theta], Nbin_a=1, Thetamax=None)

    return generate_ncov_LPLP(sigma_L, Nlens, angular_distribution)

def total_variance_LPLP(theta):
    """ 
    a function which returns the total LPLP variance as a function of theta
    (hence for use in the optimisation).
    """

    noise_sparsity = LPLP_ncov(theta) #generates the noise and sparsity as a function of theta

    noise = noise_sparsity[0] #extracts noise
    sparsity = noise_sparsity[1] #extracts sparsity
    
    variance = noise + sparsity + LPLP_ccov(theta).item() #the total variance
    
    return variance

#optimise the binning
LPLP_optimised_theta, LPLP_optimised_signal, LPLP_optimised_std, LPLP_optimised_SNR = optimise_bins(LP_primitive[0], total_variance_LPLP)

#evaluate the noise and sparsity at the optimised theta
LPLP_optimised_noise_sparsity = LPLP_ncov(LPLP_optimised_theta)

LPLP_optimised_noise = LPLP_optimised_noise_sparsity[0] #extract the optimised noise component
LPLP_optimised_sparsity = LPLP_optimised_noise_sparsity[1] #extract the optimised sparsity component
LPLP_optimised_cosmic = LPLP_ccov(LPLP_optimised_theta).item() #extract the optimised cosmic component

LPLP_dict = {'optimised_theta': LPLP_optimised_theta,
                  'optimised_signal': LPLP_optimised_signal,
                  'optimised_std': LPLP_optimised_std,
                  'optimised_SNR': LPLP_optimised_SNR,
                  'optimised_noise': LPLP_optimised_noise,
                  'optimised_sparsity': LPLP_optimised_sparsity,
                  'optimised_cosmic': LPLP_optimised_cosmic}

print(LPLP_dict)


LELE
LELE plus
{'optimised_theta': 0.001032780654894838, 'optimised_signal': 0.00020316138003101432, 'optimised_std': 1.1116856922948622e-05, 'optimised_SNR': 18.275073740638557, 'optimised_noise': 1.1398724307816555e-10, 'optimised_sparsity': 9.251142096956548e-12, 'optimised_cosmic': 3.46122670188605e-13}
LELE minus
{'optimised_theta': 0.032375860069564576, 'optimised_signal': 8.09064696259546e-06, 'optimised_std': 4.5354088874909224e-07, 'optimised_SNR': 17.83884797004789, 'optimised_noise': 1.7911859402244488e-13, 'optimised_sparsity': 2.499942777959578e-14, 'optimised_cosmic': 1.5813159652758021e-15}
LPLP
{'optimised_theta': 0.006534725574495213, 'optimised_signal': -0.00031348495195512654, 'optimised_std': 1.366199363351356e-05, 'optimised_SNR': 22.94576914354082, 'optimised_noise': 1.4871409084659655e-10, 'optimised_sparsity': 3.673974391263466e-11, 'optimised_cosmic': 1.1962352829338314e-12}


In [16]:
def optimise_bins(antiderivative, variance):
    
    redshift_distributions = load_file(f"data/redshift_distributions")
    
    def SNR(theta):

        numerator = np.abs(antiderivative(theta) - antiderivative(0))
        
        denom_sq = np.abs(variance(theta)) #this isn't great, need to check why it's like this
        
        if not np.isfinite(denom_sq):
            print(f"[BAD VAR] non-finite at theta={theta}: {denom_sq}", flush=True)
        elif denom_sq <= 0:
            print(f"[BAD VAR] non-positive at theta={theta}: {denom_sq}", flush=True)

        denominator = np.sqrt(denom_sq)

        prefactor = 2/(theta**2)
        
        return prefactor * numerator / denominator

    theta_optimal, SNR_max = find_maximum(SNR, 0, theta_max_interpolation)

    optimised_signal = 2/(theta_optimal**2)*antiderivative(theta_optimal) - antiderivative(0) #need to check
    optimised_noise = np.sqrt(np.abs(variance(theta_optimal))) #this isn't great, need to check why it's like this

    return theta_optimal, optimised_signal, optimised_noise, SNR_max         #return a single bin

In [ ]:
LPLP_optimised_theta, LPLP_optimised_signal, LPLP_optimised_std, LPLP_optimised_SNR = optimise_bins(LP_primitive[0], total_variance_LPLP)